# ATAC preprocessing: regional background, peaks, coverage

Turns the `mut_multi2.<genotype>.<cell state>` tracks from `prep/atac_tracks.ipynb` into the inputs of the gastrulation ATAC model. No figure panels.

Outputs (via `out_path()`, under `git_test/data/output/`):
- `npeaks.RDS`: joint peak set.
- `multi_type_cov.RDS`, `cov_peak_chr.RDS`, `cov_chr.RDS`: coverage statistics.
- `cgweak_intervs.RDS`: CpG-rich intervals.

These are read by `init()` in `scripts/ATAC_preprocessing.r` (where the functions live). `Fig4.ipynb` saves the model as `data/output/mod.RDS`, loaded by Fig3, Fig4, Fig5 and Supp_ATAC.

In [ ]:
source("../paths.R")
setwd(embdnmt_dir)

In [46]:
source(file.path(repo_dir, "scripts/ATAC_preprocessing.r"))
options(max.data.size = 1e+15)

## Track-writing flag

- `rewrite_tracks` guards the only write to the shared misha database: the `<track>_regcap` tracks.
- Default `F`: the tracks already exist and are only read.
- `T` overwrites them for every user of the database.

In [ ]:
# Set to TRUE to (re)write the regionally capped marginal tracks (<track>_regcap) into the shared misha track database.
# The tracks already exist, so the default only reads them.
rewrite_tracks = F

## Regional background tracks

For each marginal track (WT_lg, WT_sm, Dnmt-TKO, Dnmt-DKO, Tet-TKO), `comp_capped_regional_atac` makes `<track>_regcap`:
- coverage in a running 300 bp window,
- capped at the genome-wide 95th percentile,
- smoothed over 20 kb.

The `if (0)` cell is disabled code that deletes the `_regcap` tracks.

In [20]:
base = "mut_multi2"
marginal_trs = 
    c(sprintf("%s.%s.marginal", base, "WT_lg"),
      sprintf("%s.%s.marginal", base, "WT_sm"),
      sprintf("%s.%s.marginal", base, "DNMT_TKO_KO"),
      sprintf("%s.%s.marginal", base, "DNMT_DKO_KO"),
      sprintf("%s.%s.marginal", base, "TET_TKO_KO"))

In [21]:
if (0){
    for (tr in paste0(marginal_trs, "_regcap")){
        gtrack.rm(tr, force = T)
    }
}

In [22]:
gdb.reload()

In [ ]:
if (rewrite_tracks) {
    for (tr in marginal_trs) {
        comp_capped_regional_atac(tr)
        message(paste(tr, "done."))
    }
}

In [70]:
gdb.reload()

## Peak calling

`join_norm_peaks` calls peaks per marginal track and merges them (`gintervals.union`). Saved to `data/output/npeaks.RDS`.
- Ratio per 20 bp bin: (3 + 300 bp coverage) / (3 + 15 x regional background).
- Keeps bins above the genome-wide `T_q` quantile; contiguous bins form a peak.
- `T_q = 0.97`; 270,005 merged peaks.
- The two WT datasets are screened separately before merging.

In [28]:
peaks = join_norm_peaks()

In [31]:
saveRDS(peaks, out_path("./data/output/npeaks.RDS"))

## Coverage statistics

`collect_cov_r` records, for every genotype (KO and host of the three lines, WT_lg, WT_sm) and cell state:
- total coverage,
- coverage on peaks (overall and per chromosome),
- coverage on ribosomal-protein TSSs (-800 to +200 bp, `Rpl*`/`Rps*`): the housekeeping standard for downsampling,
- coverage per chromosome: for copy-number ratios against WT.

Writes `multi_type_cov.RDS`, `cov_peak_chr.RDS`, `cov_chr.RDS`.
- It reads peaks from the original `./data/output/npeaks.RDS`, or the copy written above under `git_test/` with `use_regenerated_inputs = TRUE` (`paths.R`).

In [71]:
collect_cov_r()

## CpG-rich intervals

`gen_cgweak_interv` keeps intervals with CpG fraction > 0.022 in a 500 bp window, with CpG content, G+C, TSS distance and mean E8.5 WT methylation. Drops chrY and chrM; saves `cgweak_intervs.RDS`.
- Used by `build_epi_hotspots_classes` to assign high, intermediate and low CpG classes.
- Caveat: `scripts/lineage_peaks_func.r` has a second `gen_cgweak_interv` that returns the table instead of saving it; the last one sourced wins.

In [39]:
gen_cgweak_interv()